# Etapa 4 · Construcción de categorías comerciales y etiquetado manual de productos

La Etapa 4 comienza una vez finalizado el análisis territorial de precios y marca el inicio del proceso de **estructuración semántica del universo de productos**. SEPA no proporciona una clasificación comercial suficientemente detallada ni homogénea para los objetivos analíticos del proyecto, por lo que se desarrollará una **clasificación propia**, diseñada para permitir comparaciones consistentes entre cadenas y habilitar análisis posteriores basados en NLP y Machine Learning.

**Objetivo general**  
Construir un sistema de categorías reproducible, claro y coherente que permita asignar cada producto a un rubro comercial definido, sirviendo como base para la futura clasificación automática.

### Pasos principales de la etapa

**Definición de categorías comerciales**  
Se establecerá un conjunto de categorías que representen adecuadamente el universo de productos. Estas categorías deberán ser:  
- mutuamente excluyentes,  
- exhaustivas,  
- consistentes con la estructura comercial real,  
- adecuadas para análisis comparativos entre cadenas.

**Establecimiento de criterios de asignación**  
Se definirán reglas claras para determinar a qué categoría pertenece cada producto. Los criterios deberán ser:  
- simples de aplicar,  
- replicables por distintos analistas,  
- compatibles con la futura automatización mediante modelos de NLP y Machine Learning.

**Selección y etiquetado manual de una muestra de productos**  
Se construirá un conjunto inicial de productos etiquetados manualmente. Esta muestra funcionará como:  
- referencia para revisar la coherencia de las categorías,  
- base inicial para experimentar con la clasificación automática en la Etapa 5,  
- referencia para evaluar la consistencia de las etiquetas entre productos de ambas cadenas.

**Resultado esperado**  
Al finalizar la Etapa 4 se contará con:  
- un esquema de categorías definido,  
- criterios de asignación documentados,  
- un conjunto inicial de productos etiquetados manualmente,  
- una base preparada para avanzar hacia la clasificación automática mediante técnicas de NLP y Machine Learning en la Etapa 5.


## 1. Carga de datos

En este apartado se cargan los datasets necesarios para iniciar la construcción de la clasificación comercial propia.  
Los datos utilizados provienen del universo depurado de siete provincias y contienen la información consolidada de productos, precios, descripciones y marcas reportadas por cada cadena.

El objetivo de esta sección es disponer de un dataset limpio y estructurado que permita avanzar hacia la definición de categorías y el etiquetado manual.  
La carga se realiza en formato Parquet para asegurar eficiencia en lectura y preservación de tipos de datos.


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [2]:
# Cargamos el dataset de precios procesado de las 7 provincias
dataset_precios_7_provincias = pd.read_parquet('../data/processed/dataset_precios_7_provincias.parquet')

In [3]:
# Verificamos la estructura del dataset
dataset_precios_7_provincias

,id_sucursal,id_producto,productos_descripcion,productos_marca,precio_efectivo,provincia,sucursales_localidad,sucursales_latitud,sucursales_longitud,id_comercio,cadena
0,1,40084107,HUEVO DE CHOCOLATE,KINDE,2785.0,CABA,CAPITAL FEDERAL,-34.597137,-58.497643,15,DIA
1,1,42277057,CREM FACIAL CUID NUT,NIVEA,6455.0,CABA,CAPITAL FEDERAL,-34.597137,-58.497643,15,DIA
2,1,75079437,ANTITRANS CREM WOMEN,DOVE,12750.0,CABA,CAPITAL FEDERAL,-34.597137,-58.497643,15,DIA
3,1,75082734,ANTITRANS ORIG BARRA,DOVE,8019.0,CABA,CAPITAL FEDERAL,-34.597137,-58.497643,15,DIA
4,1,75083694,DEO BLUE LAVEN BARRA,AXE,8699.0,CABA,CAPITAL FEDERAL,-34.597137,-58.497643,15,DIA
...,...,...,...,...,...,...,...,...,...,...,...
2912833,222,7790040143364,GALLETITAS SURTIDO DIVERSION BOLSA X 400 GRS,DIVERSION,2720.0,CORDOBA,CORDOBA,-31.423396,-64.179534,10,CARREFOUR
2912834,526,7794000007109,MAYONESA LIVIANA HELLMANNS DOY PACK X 475 GRS,HELLMANNS,3089.0,CABA,CIUDAD AUTONOMA DE BUENOS AIRES,-34.609566,-58.443746,10,CARREFOUR
2912835,371,7791813888376,GASEOSA COLA REGULAR PEPSI N LATA X 354 CC,PEPSI,2205.0,CABA,CIUDAD AUTONOMA DE BUENOS AIRES,-34.579005,-58.490393,10,CARREFOUR
2912836,684,7792180140708,HARINA LEUDANTE C VIT DPUREZA X 1 KG,PUREZA,1959.0,CABA,CIUDAD AUTONOMA DE BUENOS AIRES,-34.615549,-58.387978,10,CARREFOUR


## 2. Construcción del universo de productos comparables

En este punto se integran los elementos necesarios para conformar un **dataset único de productos**, donde cada fila representa un ítem comercial con información homogénea entre cadenas. El objetivo es obtener una estructura consistente que permita avanzar hacia la clasificación comercial y los análisis posteriores.

**Cálculo del precio medio por cadena y producto**  
A partir del universo depurado de siete provincias, se calcula el **precio medio** de cada producto en cada cadena. Este promedio se obtiene agregando todas las observaciones disponibles por sucursal y luego reorganizando la información en formato ancho, con una columna para DIA y otra para Carrefour.

**Extracción de descripciones y marca por cadena**  
SEPA reporta descripciones y marcas por producto y sucursal. Para construir un universo único, se toma una descripción representativa por cadena y producto, manteniendo la estructura original reportada por SEPA. Se extraen:  
- la descripción de DIA,  
- la descripción de Carrefour,  
- la marca reportada por Carrefour, ya que incluir la marca de ambas cadenas generaría duplicación de información y ruido, sin aportar mejoras significativas al modelo.  
Estas variables funcionan como atributos base para la futura clasificación comercial.

**Integración de atributos y precios en un único dataset**  
Finalmente, se combinan los precios medios y los atributos de producto mediante uniones por `id_producto`. El resultado es un dataset consolidado con:  
- precio medio en DIA,  
- precio medio en Carrefour,  
- descripción DIA,  
- descripción Carrefour,  
- marca Carrefour.  

Este dataset constituye el **universo de productos comparables**, que será utilizado en la siguiente etapa para la construcción de categorías comerciales y el etiquetado manual.



In [4]:
# Comprobamos la cantidad de descripciones distintas por producto y cadena
variacion_descripciones = (
    dataset_precios_7_provincias
    .groupby(['id_producto', 'cadena'])['productos_descripcion']
    .nunique(dropna=True)
)

print(
    'Combinaciones producto-cadena con varias descripciones:',
    (variacion_descripciones > 1).sum()
)

Combinaciones producto-cadena con varias descripciones: 0


In [5]:
# Comprobamos la cantidad de marcas distintas por producto y cadena
variacion_marcas = (
    dataset_precios_7_provincias
    .groupby(['id_producto', 'cadena'])['productos_marca']
    .nunique(dropna=True)
)

print(
    'Combinaciones producto-cadena con varias marcas:',
    (variacion_marcas > 1).sum()
)

Combinaciones producto-cadena con varias marcas: 0


Se comprobó que no existen múltiples descripciones ni marcas distintas para una misma combinación de producto y cadena dentro del dataset analizado. Por este motivo, se puede seleccionar un registro por combinación para construir el dataset consolidado, manteniendo la trazabilidad de la información original.

In [6]:
# Calculamos precio medio por cadena y producto
precios_medios = (
    dataset_precios_7_provincias
    .groupby(['cadena', 'id_producto'])['precio_efectivo']
    .mean()
    .reset_index()
    .pivot(index='id_producto', columns='cadena', values='precio_efectivo')
    .rename(columns={'DIA': 'precio_medio_dia', 'CARREFOUR': 'precio_medio_carrefour'}).round(2)
    .reset_index()
)

# Extraemos descripciones y marca por cadena
info_productos = (
    dataset_precios_7_provincias[
        ['id_producto', 'cadena', 'productos_descripcion', 'productos_marca']
    ]
    .drop_duplicates(['id_producto', 'cadena'])
)

# Descripción DIA
desc_dia = (
    info_productos[info_productos['cadena'] == 'DIA']
    .rename(columns={'productos_descripcion': 'descripcion_dia'})
    [['id_producto', 'descripcion_dia']]
)

# Descripción y marca Carrefour
desc_carrefour = (
    info_productos[info_productos['cadena'] == 'CARREFOUR']
    .rename(columns={'productos_descripcion': 'descripcion_carrefour'})
    [['id_producto', 'descripcion_carrefour']]
)

marca_carrefour = (
    info_productos[info_productos['cadena'] == 'CARREFOUR']
    .rename(columns={'productos_marca': 'productos_marca'})
    [['id_producto', 'productos_marca']]
)

# Unimos todo en un único dataset final
media_productos_comunes = (
    precios_medios
    .merge(desc_dia, on='id_producto', how='left')
    .merge(desc_carrefour, on='id_producto', how='left')
    .merge(marca_carrefour, on='id_producto', how='left')
)

media_productos_comunes.head()


,id_producto,precio_medio_carrefour,precio_medio_dia,descripcion_dia,descripcion_carrefour,productos_marca
0,40084107,2875.50,2793.33,HUEVO DE CHOCOLATE,HUEVO CHOCOLATE SORPRESA CELESTE KINDER X 20 GRS,KINDER
1,42277057,7872.42,6455.00,CREM FACIAL CUID NUT,CREMA FACIAL CUIDADO NUTRITIVO NIVEA CARE X 100CC,NIVEA
2,75079437,18265.21,12750.00,ANTITRANS CREM WOMEN,ANTITRANSP FEM CREMA CLINICAL SOFT DOVE X 58GRS,DOVE
3,75082734,7906.62,8037.19,ANTITRANS ORIG BARRA,ANTITRANSPIRANTE BARRA ORIGINAL DOVE X 53 GR,DOVE
4,75083694,8677.08,8718.77,DEO BLUE LAVEN BARRA,DESODORANTE STICK BLUE LAVENDER AXE X 58 GRM,AXE


La salida confirma que la integración se realizó correctamente y que el dataset resultante contiene la información necesaria para avanzar hacia el etiquetado manual y la posterior etapa de clasificación mediante técnicas de NLP.


## 3. Definición del sistema de clasificación de productos

**Criterio de clasificación**

Para organizar el catálogo y preparar los datos para la futura clasificación automática mediante NLP y Machine Learning, se definió un sistema de clasificación compuesto por cinco categorías principales. La estructura busca simplificar el etiquetado manual, mantener criterios consistentes entre cadenas y conservar una utilidad comercial para el análisis comparativo de productos. Las categorías se diseñaron considerando la naturaleza y finalidad principal de cada producto, buscando que sean prácticas y aplicables tanto a DIA como a Carrefour.

**Categorías definidas**

| Categoría | Descripción | Ejemplos |
|---|---|---|
| **Alimentos** | Productos destinados al consumo humano, incluyendo alimentos frescos, refrigerados, congelados, secos, conservables y golosinas. | Carnes, pollo, frutas, verduras, pan, lácteos, arroz, fideos, harina, galletitas, cereales, chocolates, alfajores, caramelos. |
| **Bebidas** | Productos destinados principalmente al consumo como bebida, independientemente de su formato de venta o preparación. | Agua, gaseosas, jugos, cerveza, vino, energéticas, jugo en polvo. |
| **Perfumería y limpieza** | Productos de higiene corporal, cuidado personal y limpieza del hogar. | Shampoo, jabón, pasta dental, desodorante, pañales, higiene femenina, papel higiénico, rollo de cocina, detergente, lavandina, suavizante, limpiadores. |
| **Mascotas** | Productos destinados a la alimentación, higiene y cuidado de animales domésticos. | Alimento balanceado, snacks para perros y gatos, arena sanitaria y accesorios básicos para mascotas. |
| **Otros** | Productos que no encajan en las categorías principales, generalmente artículos complementarios, accesorios o productos de uso general. | Pilas, juguetes, descartables (vasos, platos, cubiertos), bombillas, film, aluminio, fósforos, encendedores, bolsas, servilletas especiales y artículos de bazar. |

**Reglas de asignación**

- Cada producto se asigna a una única categoría según su naturaleza y finalidad principal.
- Todos los productos alimenticios destinados al consumo humano se clasifican en **Alimentos**, independientemente de su conservación o presentación. Esto incluye alimentos frescos, refrigerados, congelados, secos, en polvo, conservas, golosinas y panificados.
- Los lácteos, incluida la leche líquida y la leche en polvo, se clasifican en **Alimentos**.
- Las bebidas incluyen productos comercializados principalmente para ser bebidos. Los jugos en polvo se clasifican en **Bebidas**, aunque se vendan en formato sólido.
- Los productos de higiene corporal, cuidado personal, limpieza del hogar y absorbentes se agrupan en **Perfumería y limpieza**.
- Los productos destinados a la alimentación, higiene y cuidado de animales domésticos se clasifican en **Mascotas**.
- Los artículos descartables, de bazar y de uso general que no correspondan a las categorías anteriores se clasifican en **Otros**.
- En los productos de uso doméstico que puedan generar dudas, se considera su finalidad principal y se aplica el mismo criterio de forma consistente en todo el catálogo.
- Los criterios se aplican de manera uniforme a los productos de ambas cadenas.

**Justificación de la estructura**

La estructura de clasificación se diseñó para equilibrar simplicidad, coherencia semántica y utilidad práctica en el análisis de productos. Cada categoría agrupa artículos según su finalidad y tipo de uso, procurando que el sistema sea claro para el etiquetado manual y aplicable a las siguientes etapas del proyecto.

La categoría **Alimentos** reúne productos destinados al consumo humano, abarcando alimentos frescos, secos, conservables y golosinas. Esta integración permite trabajar con una categoría amplia que representa una parte importante del catálogo.

**Perfumería y limpieza** agrupa artículos de higiene personal y limpieza del hogar para simplificar la clasificación y trabajar con un rubro comercial más amplio. Aunque ambos segmentos tienen finalidades diferentes, se mantienen unidos en esta etapa como una decisión metodológica que podrá revisarse si los resultados de clasificación o las necesidades del análisis lo justifican.

**Bebidas** se mantiene como categoría independiente debido a su coherencia semántica y a la presencia de productos con características y descripciones comerciales diferenciadas dentro del catálogo.

La categoría **Mascotas** conserva su identidad propia, ya que los productos destinados a animales domésticos presentan características y terminología específicas que los distinguen de los productos para consumo humano y de los artículos de uso general.

Finalmente, **Otros** funciona como una categoría residual para los productos que no encajan en las categorías principales. Debido a la variedad de artículos que puede reunir, su consistencia deberá revisarse durante la evaluación del modelo de clasificación.

Esta estructura reduce la cantidad de categorías y facilita el etiquetado manual, manteniendo una distinción entre los principales rubros comerciales del catálogo. Su adecuación para la clasificación automática se evaluará en la **Etapa 5: Categorización de productos mediante NLP y Machine Learning**, considerando las métricas generales, el desempeño individual de cada categoría y la revisión de los errores del modelo.


## 4. Preparación del dataset para el etiquetado manual

Para iniciar el proceso de clasificación comercial, se añadió al dataset una columna destinada a registrar la categoría asignada a cada producto. Esta columna se dejó vacía para permitir el etiquetado manual fuera del entorno de programación.

El dataset completo se exportó en formato Excel, lo que facilita la revisión de descripciones, la comparación entre cadenas y la asignación de categorías según los criterios definidos en el sistema de clasificación. Este archivo constituye la base del conjunto de entrenamiento que se utilizará en la etapa siguiente.

Una vez completado el etiquetado manual, el archivo se vuelve a cargar en el entorno de análisis para verificar la consistencia de las categorías asignadas y asegurar que todos los productos hayan sido clasificados correctamente. Esta verificación es fundamental para garantizar la calidad del conjunto de datos que se empleará en la futura clasificación automática mediante técnicas de NLP y Machine Learning.


In [7]:
# Creamos la columna vacía para cargar la categoría manualmente
media_productos_comunes['categoria'] = ""

In [8]:
# Exportamos a Excel para etiquetar manualmente
media_productos_comunes.to_excel("../data/processed/productos_etiquetar.xlsx", index=False)
print('Datasets guardados en ../data/processed/')

Datasets guardados en ../data/processed/


In [9]:
# Cargamos el dataset con 400 productos etiquetados manualmente y los restantes pendientes de clasificación
productos_etiquetados_parcialmente = pd.read_excel("../data/processed/productos_etiquetados_parcialmente.xlsx",sheet_name="Sheet1")

In [10]:
# Verificamos la correcta lectura del dataset
productos_etiquetados_parcialmente

,id_producto,precio_medio_carrefour,precio_medio_dia,descripcion_dia,descripcion_carrefour,productos_marca,categoria
0,40084107,2875.50,2793.33,HUEVO DE CHOCOLATE,HUEVO CHOCOLATE SORPRESA CELESTE KINDER X 20 GRS,KINDER,ALIMENTOS
1,42277057,7872.42,6455.00,CREM FACIAL CUID NUT,CREMA FACIAL CUIDADO NUTRITIVO NIVEA CARE X 100CC,NIVEA,NaN
2,75079437,18265.21,12750.00,ANTITRANS CREM WOMEN,ANTITRANSP FEM CREMA CLINICAL SOFT DOVE X 58GRS,DOVE,NaN
3,75082734,7906.62,8037.19,ANTITRANS ORIG BARRA,ANTITRANSPIRANTE BARRA ORIGINAL DOVE X 53 GR,DOVE,NaN
4,75083694,8677.08,8718.77,DEO BLUE LAVEN BARRA,DESODORANTE STICK BLUE LAVENDER AXE X 58 GRM,AXE,PERFUMERIA Y LIMPIEZA
...,...,...,...,...,...,...,...
2215,8445291749245,4680.76,4558.57,CAFE DOLCA CAPUCCINO,CAFE CREMOSO CAPPUCCINO INSTANT DOLCA X 100 GRS,DOLCA,NaN
2216,8445291749306,5609.40,5636.75,CAFE CHOCO/MOCHA CRE,CAFE CREMOSO CHOCOMOCHA INSTANT DOLCA X 125 GRS,DOLCA,NaN
2217,8445292738200,12428.96,12539.32,CAFE NESCAFE SUAVE,CAFE INSTANTANEO SUAVE ORIGENES DP DOLCA X 160...,DOLCA,NaN
2218,9002490100070,3529.40,3447.08,BEBIDA ENERGIZANTE,BEBIDA ENERGIZANTE RED BULL LATA X 250 CC.,RED BULL,NaN


## 5. Validación de categorías y control de calidad del etiquetado

En esta etapa se verifica la consistencia formal del etiquetado manual realizado sobre el universo de productos.

Primero se define la lista oficial de categorías válidas y se normalizan las etiquetas para evitar errores por espacios o formatos. Luego se identifica cuántos productos tienen una categoría asignada y cuántos permanecen pendientes.

Se controla que todas las etiquetas asignadas pertenezcan a la taxonomía definida, detectando posibles valores incorrectos o inconsistentes. Finalmente, se obtiene la distribución de productos por categoría para conocer la composición del conjunto etiquetado e identificar posibles desequilibrios que deban considerarse durante el entrenamiento y la evaluación del modelo de clasificación.


In [11]:
# Categorías válidas definidas en la taxonomía
categorias_validas = [
    'ALIMENTOS', 'BEBIDAS',
    'PERFUMERIA Y LIMPIEZA',
    'MASCOTAS', 'OTROS'
]

# Normalizamos etiquetas y detectamos asignación
categorias = productos_etiquetados_parcialmente['categoria'].astype('string').str.strip()
tiene_categoria = categorias.notna() & categorias.ne('')

print(f"Productos totales: {len(productos_etiquetados_parcialmente)}")
print(f"Productos etiquetados: {tiene_categoria.sum()}")
print(f"Productos pendientes: {(~tiene_categoria).sum()}")

# Etiquetas fuera de la taxonomía
etiquetas_invalidas = categorias[tiene_categoria].loc[lambda s: ~s.isin(categorias_validas)].unique()
print("Etiquetas no reconocidas:", etiquetas_invalidas)

# Distribución de categorías
print("\nDistribución de categorías:")
print(categorias[tiene_categoria].value_counts().sort_index())


Productos totales: 2220
Productos etiquetados: 400
Productos pendientes: 1820
Etiquetas no reconocidas: <StringArray>
[]
Length: 0, dtype: string

Distribución de categorías:
categoria
ALIMENTOS                184
BEBIDAS                   62
MASCOTAS                  27
OTROS                     12
PERFUMERIA Y LIMPIEZA    115
Name: count, dtype: Int64


La salida confirma que el proceso de etiquetado manual se realizó sobre una parte del universo de productos y que las etiquetas asignadas coinciden con la taxonomía definida.

De los **2.220 productos totales**, se etiquetaron manualmente **400 (18%)**, mientras que **1.820 permanecen pendientes de clasificación**.

No se detectaron **etiquetas inválidas**, lo que indica que todos los valores asignados coinciden con alguna de las cinco categorías permitidas. Este control verifica la validez formal de las etiquetas, aunque no garantiza por sí solo que cada producto haya sido clasificado semánticamente de manera correcta.

La distribución muestra que las categorías principales cuentan con una mayor cantidad de ejemplos, mientras que *Mascotas* y *Otros* presentan una representación menor. Esta diferencia deberá tenerse en cuenta al preparar el conjunto de entrenamiento y evaluación del modelo de clasificación.

El etiquetado manual constituye una base inicial para experimentar con la clasificación automática en la Etapa 5. Antes del entrenamiento, será necesario revisar la consistencia de las etiquetas y evaluar si las categorías con menor representación requieren ejemplos adicionales.
